In [1]:
from QuantConnect import *

qb = QuantBook()

20260426 12:44:12.514 TRACE:: QuantBook started; Is Python: True
20260426 12:44:12.997 TRACE:: AlgorithmManager.CreateTokenBucket(): Initializing LeakyBucket: Capacity: 120 RefillAmount: 18 TimeInterval: 1440
20260426 12:44:12.999 TRACE:: LocalObjectStore.Initialize(): Storage Root: /Users/arifkhan/github/Lean/Launcher/bin/Debug/storage. StorageFileCount 10000. StorageLimit 10240MB. StoragePermissions Read=True Write=True Delete=True
20260426 12:44:13.007 TRACE:: HistoryProviderManager.Initialize(): history providers [SubscriptionDataReaderHistoryProvider]
20260426 12:44:13.012 TRACE:: AlgorithmManager.CreateTokenBucket(): Initializing LeakyBucket: Capacity: 120 RefillAmount: 18 TimeInterval: 1440
20260426 12:44:13.012 TRACE:: LocalObjectStore.Initialize(): Storage Root: /Users/arifkhan/github/Lean/Launcher/bin/Debug/storage. StorageFileCount 10000. StorageLimit 10240MB. StoragePermissions Read=True Write=True Delete=True
20260426 12:44:13.012 TRACE:: HistoryProviderManager.Initialize(

In [7]:
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Fetch SPY historical data
spy = qb.AddEquity("SPY", Resolution.Daily)
history = qb.History([spy.Symbol], 500, Resolution.Daily)



In [11]:
history

""


In [ ]:
# Extract close prices
close_col = "close" if "close" in history.columns else "Close"
# Handle empty history and column-name differences safely
if history.empty:
    history = qb.History(spy.Symbol, 500, Resolution.Daily)

if history.empty:
    raise ValueError("No SPY history returned from qb.History().")

if "close" in history.columns:
    close_col = "close"
elif "Close" in history.columns:
    close_col = "Close"
else:
    raise KeyError(f"No close column found. Available columns: {list(history.columns)}")

# Support both MultiIndex and single-index history shapes
if isinstance(history.index, pd.MultiIndex):
    close_prices = history[close_col].unstack(level=0)
else:
    close_prices = history[[close_col]].copy()
close_prices.columns = ["SPY"]
close_prices.index = pd.to_datetime(close_prices.index)

# Calculate EMA 9 and EMA 15
close_prices["EMA9"] = close_prices["SPY"].ewm(span=9, adjust=False).mean()
close_prices["EMA15"] = close_prices["SPY"].ewm(span=15, adjust=False).mean()

# Generate signals: Buy when EMA9 crosses above EMA15, Sell when below
close_prices["Signal"] = 0
close_prices.loc[close_prices["EMA9"] > close_prices["EMA15"], "Signal"] = 1
close_prices.loc[close_prices["EMA9"] < close_prices["EMA15"], "Signal"] = -1

# Calculate strategy returns
close_prices["Returns"] = close_prices["SPY"].pct_change()
close_prices["StrategyReturns"] = close_prices["Signal"].shift(1) * close_prices["Returns"]
close_prices["CumulativeMarket"] = (1 + close_prices["Returns"]).cumprod()
close_prices["CumulativeStrategy"] = (1 + close_prices["StrategyReturns"]).cumprod()

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [3, 1, 2]})
fig.suptitle("SPY EMA9 / EMA15 Crossover Strategy", fontsize=16, fontweight="bold")

# --- Price + EMAs ---
axes[0].plot(close_prices.index, close_prices["SPY"], label="SPY Price", color="steelblue", lw=1.5)
axes[0].plot(close_prices.index, close_prices["EMA9"], label="EMA 9", color="orange", lw=1.2, linestyle="--")
axes[0].plot(close_prices.index, close_prices["EMA15"], label="EMA 15", color="green", lw=1.2, linestyle="--")

# Buy/Sell markers
buy_signals = close_prices[
    (close_prices["Signal"] == 1) & (close_prices["Signal"].shift(1) != 1)
]
sell_signals = close_prices[
    (close_prices["Signal"] == -1) & (close_prices["Signal"].shift(1) != -1)
]
axes[0].scatter(buy_signals.index, buy_signals["SPY"], marker="^", color="lime", s=80, zorder=5, label="Buy Signal")
axes[0].scatter(sell_signals.index, sell_signals["SPY"], marker="v", color="red", s=80, zorder=5, label="Sell Signal")
axes[0].set_ylabel("Price ($)")
axes[0].legend(loc="upper left", fontsize=8)
axes[0].grid(True, alpha=0.3)

# --- Signal ---
axes[1].fill_between(close_prices.index, close_prices["Signal"], 0,
                     where=close_prices["Signal"] > 0, color="lime", alpha=0.5, label="Long")
axes[1].fill_between(close_prices.index, close_prices["Signal"], 0,
                     where=close_prices["Signal"] < 0, color="red", alpha=0.5, label="Short")
axes[1].set_ylabel("Signal")
axes[1].legend(loc="upper left", fontsize=8)
axes[1].grid(True, alpha=0.3)

# --- Cumulative Returns ---
axes[2].plot(close_prices.index, close_prices["CumulativeStrategy"], label="EMA Strategy", color="darkorange", lw=1.5)
axes[2].plot(close_prices.index, close_prices["CumulativeMarket"], label="Buy & Hold", color="steelblue", lw=1.5, linestyle="--")
axes[2].set_ylabel("Cumulative Return")
axes[2].legend(loc="upper left", fontsize=8)
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

# Print summary stats
total_return = close_prices["CumulativeStrategy"].iloc[-1] - 1
market_return = close_prices["CumulativeMarket"].iloc[-1] - 1
print(f"Strategy Total Return : {total_return:.2%}")
print(f"Market  Total Return  : {market_return:.2%}")